# 05 - Model Training (Member 3 - Tharakie D.P.S.D.)
**Project:** ML-based prediction of cardio-metabolic risk (metabolic syndrome) from NHANES 2011-2014 wrist-accelerometer data.

| Task | What this notebook does |
|---|---|
| **M3-01** | Uses the fixed stratified 80/20 train/test split (`split` column, seed 42) |
| **M3-02 / 04 / 05** | Trains Logistic Regression, Random Forest, Gradient Boosting |
| **M3-06** | Leakage-safe pipeline (impute -> scale -> classifier) tuned with GridSearchCV in stratified 5-fold CV |
| **M3-08** | Trains every model on Feature Sets 0, A and B and compares them |

**Leakage rules:** the test set is never used in this notebook's tuning; imputer and scaler are fitted inside each CV training fold only.
Code lives in `src/models/classifiers.py` (models + grids) and `src/models/pipeline.py` (loading, pipeline, tuning).

## 1. Setup

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root = nearest parent folder that contains src/models
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src" / "models").is_dir())
sys.path.insert(0, str(ROOT))

from src.models.classifiers import get_models, get_param_grids
from src.models.pipeline import FEATURE_SET_FILES, load_feature_set, tune_model

FEATURE_SETS = list(FEATURE_SET_FILES)        # ['Set 0', 'Set A', 'Set B']
MODELS = list(get_models())                   # ['Logistic Regression', 'Random Forest', 'Gradient Boosting']
print("Project root:", ROOT)
print("Feature sets:", FEATURE_SETS)
print("Models      :", MODELS)

## 2. Load the feature sets (M3-01)
All three sets use the **same 4,575 participants** and the **same train/test split**, so the comparison is fair.

In [ ]:
data = {}
rows = []
for s in FEATURE_SETS:
    X_train, X_test, y_train, y_test = load_feature_set(s)
    data[s] = (X_train, X_test, y_train, y_test)
    rows.append({"feature_set": s, "n_features": X_train.shape[1],
                 "n_train": len(X_train), "n_test": len(X_test),
                 "risk_rate_train": round(y_train.mean(), 3),
                 "risk_rate_test": round(y_test.mean(), 3),
                 "missing_values_train": int(X_train.isna().sum().sum())})
display(pd.DataFrame(rows).set_index("feature_set"))

## 3. Hyperparameter grids
`GridSearchCV` tries every combination below. Each one is scored with stratified 5-fold CV (ROC-AUC).

In [ ]:
grid_rows = []
for m, grid in get_param_grids().items():
    n = 1
    for values in grid.values():
        n *= len(values)
    grid_rows.append({"model": m, "combinations": n, "fits (x5 folds)": n * 5,
                      "grid": {k.replace("clf__", ""): v for k, v in grid.items()}})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(grid_rows).set_index("model"))

## 4. Train and tune: 3 models x 3 feature sets (M3-06, M3-08)
For every (set, model) pair: grid search on the **training data only**, then the best pipeline is refitted on the full training set.
This takes a few minutes.

In [ ]:
searches = {}      # (feature_set, model) -> fitted GridSearchCV
results = []

for s in FEATURE_SETS:
    X_train, _, y_train, _ = data[s]
    for m in MODELS:
        t0 = time.time()
        search = tune_model(m, X_train, y_train)
        seconds = time.time() - t0
        searches[(s, m)] = search

        best = search.best_index_
        cv = search.cv_results_
        results.append({
            "feature_set": s,
            "model": m,
            "cv_roc_auc_mean": cv["mean_test_score"][best],
            "cv_roc_auc_std": cv["std_test_score"][best],
            "best_params": {k.replace("clf__", ""): v for k, v in search.best_params_.items()},
            "tuning_seconds": round(seconds, 1),
        })
        print(f"{s:6s} | {m:20s} | CV ROC-AUC = {cv['mean_test_score'][best]:.3f} "
              f"+/- {cv['std_test_score'][best]:.3f} | {seconds:5.1f}s")

cv_results = pd.DataFrame(results)

## 5. Results: cross-validated ROC-AUC (training data only)

In [ ]:
table = cv_results.pivot(index="model", columns="feature_set", values="cv_roc_auc_mean")
table = table.loc[MODELS, FEATURE_SETS]
display(table.style.format("{:.3f}").highlight_max(axis=None, color="#c6efce")
        .set_caption("Mean 5-fold CV ROC-AUC (best hyperparameters)"))

display(cv_results[["feature_set", "model", "best_params", "tuning_seconds"]])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.25
for i, m in enumerate(MODELS):
    sub = cv_results[cv_results["model"] == m].set_index("feature_set").loc[FEATURE_SETS]
    xs = [j + (i - 1) * width for j in range(len(FEATURE_SETS))]
    bars = ax.bar(xs, sub["cv_roc_auc_mean"], width, yerr=sub["cv_roc_auc_std"],
                  capsize=3, label=m)
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set_xticks(range(len(FEATURE_SETS)), FEATURE_SETS)
ax.set_ylim(0.5, 0.9)
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_ylabel("5-fold CV ROC-AUC")
ax.set_title("Model x feature set comparison (training data, mean +/- std)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 6. How to read these results
- **Set 0 -> Set A:** gain from adding basic wearable activity data (RQ2).
- **Set A -> Set B:** gain from feature engineering (intensity, sedentary, rhythm, PCA, K-means).
- **Between models:** whether non-linear ensembles (RF, GB) beat the linear baseline (LR).
- These are **cross-validation** scores on training data. The final, unbiased numbers come from the held-out test set (Part 4 / Member 4).